# **Chatbot FAQ Akademik dengan Sentence-BERT**
## Tugas Pengganti UAS Kecerdasan Buatan - 01
---

**Kelompok:** Rabu-18

**Anggota:**
- RAKA ARRAYAN MUTTAQIEN (2306161800)
- DAFFA HARDHAN (2306161763)
- WILMAN SARAGIH SITIO (2306161776)

---

Notebook ini mendokumentasikan alur lengkap pembangunan chatbot FAQ akademik berbasis **semantic retrieval**. Fokus utamanya bukan generasi jawaban baru, tetapi memilih FAQ yang paling relevan secara makna dari knowledge base yang sudah disusun.

Struktur notebook dibuat berurutan agar mudah diikuti pembaca:
1. instalasi dan pemuatan data,
2. preprocessing teks,
3. pembangunan baseline Sentence-BERT,
4. evaluasi retrieval,
5. fine-tuning ringan,
6. analisis error dan threshold,
7. penyimpanan model untuk deployment.

Saat membaca notebook ini, perhatikan terutama tiga jenis output:
- tabel ukuran data dan contoh data,
- metrik evaluasi seperti Top-1 Accuracy, Top-3 Accuracy, dan MRR,
- tabel error analysis dan threshold tuning yang dipakai untuk pembahasan laporan.


## 1. Install Library

Bagian ini memasang dependensi minimum yang dibutuhkan notebook. Library yang dipakai dipilih agar tetap ringan dan mudah dijalankan di Colab:
- `sentence-transformers` untuk model SBERT dan training API,
- `datasets` untuk membungkus data train/validation,
- `scikit-learn` untuk metrik evaluasi,
- `pandas` dan `numpy` untuk manipulasi data,
- `torch` sebagai backend model.

Jika runtime Colab masih bersih, jalankan cell install ini terlebih dahulu lalu lanjutkan ke section berikutnya.

---


In [22]:
!pip install -q sentence-transformers datasets scikit-learn pandas numpy torch

## 2. Download atau Load Dataset

Section ini memastikan semua file CSV tersedia di environment Colab. Notebook mendukung dua skenario:
- file sudah di-upload atau di-mount ke Colab,
- file diunduh lebih dulu melalui URL eksternal.

Empat file utama yang dipakai adalah:
- `faq.csv` untuk knowledge base FAQ,
- `similarity_pairs.csv` untuk fine-tuning semantic similarity,
- `faq_eval_queries.csv` untuk evaluasi retrieval in-domain,
- `faq_ood_queries.csv` untuk analisis fallback out-of-domain.

Tujuan section ini adalah memastikan eksperimen bisa direproduksi tanpa asumsi tersembunyi tentang lokasi data.

---


In [23]:
# Aktifkan jika perlu download dari Dropbox
!wget -O faq.csv "https://www.dropbox.com/scl/fi/8nkmuvutt1qtw3z1p42ta/faq.csv?rlkey=fcogoflo9uax6c3fw3h8wa7dk&st=n4qlix6v&dl=1"
!wget -O similarity_pairs.csv "https://www.dropbox.com/scl/fi/wla5qpjvanhu5v628hesa/similarity_pairs.csv?rlkey=6ukcv8jn8d8jv9pkc89hubnmo&st=5ab84d1x&dl=1"
!wget -O faq_eval_queries.csv "https://www.dropbox.com/scl/fi/iqvawagz9wc8e0me9llm9/faq_eval_queries.csv?rlkey=twehallso2vzjp7bzh80htlb6&st=b3o6ipyh&dl=1"
!wget -O faq_ood_queries.csv "https://www.dropbox.com/scl/fi/4lnkg3guorc9tubyzqog5/faq_ood_queries.csv?rlkey=uizdcbzrrxxlx2f5i08awnusp&st=uzkoiedk&dl=1"

--2026-05-24 08:56:44--  https://www.dropbox.com/scl/fi/8nkmuvutt1qtw3z1p42ta/faq.csv?rlkey=fcogoflo9uax6c3fw3h8wa7dk&st=n4qlix6v&dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.6.18, 2620:100:6019:18::a27d:412
Connecting to www.dropbox.com (www.dropbox.com)|162.125.6.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ucab0c58386753f28ea5c5dd38fd.dl.dropboxusercontent.com/cd/0/inline/DBFwrIgoRDQyZNPxqrZi4l_st-BxSUpd5BGJQ3jBswt5P-MddZFwcG7C2v-FguT11et52UrM-GPTij9azJNJ3VeXocRwYuDmcbm8GVIWw_KFrt_e0Y_m0lLR3gjVLwC8uh7bXsU6N0s_SP46RG7Pwc0Z/file?dl=1# [following]
--2026-05-24 08:56:44--  https://ucab0c58386753f28ea5c5dd38fd.dl.dropboxusercontent.com/cd/0/inline/DBFwrIgoRDQyZNPxqrZi4l_st-BxSUpd5BGJQ3jBswt5P-MddZFwcG7C2v-FguT11et52UrM-GPTij9azJNJ3VeXocRwYuDmcbm8GVIWw_KFrt_e0Y_m0lLR3gjVLwC8uh7bXsU6N0s_SP46RG7Pwc0Z/file?dl=1
Resolving ucab0c58386753f28ea5c5dd38fd.dl.dropboxusercontent.com (ucab0c58386753f28ea5c5dd38fd.dl.dropboxusercontent.com)..

In [24]:
# Import library utama yang dipakai di seluruh notebook.
import random
import re

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    util,
)
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sklearn.metrics import accuracy_score

# Konfigurasi global dibuat di satu tempat agar mudah direvisi dan konsisten.
SEED = 42
BASE_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
THRESHOLD = 0.55

# Seed dipasang untuk menjaga hasil eksperimen tetap reproducible.
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


In [25]:
faq_df = pd.read_csv('faq.csv')
faq_df = faq_df.rename(columns={'pertanyaan': 'question', 'jawaban': 'answer'})
faq_df['question'] = faq_df['question'].astype(str)
faq_df['answer'] = faq_df['answer'].astype(str)
faq_df.head()

,question,answer
0,Apa saja yang dinilai oleh supervisor perusaha...,Supervisor perusahaan dan dosen penguji akan m...
1,Bagaimana aturan berpakaian saat sidang KP?,Mahasiswa wajib mengenakan pakaian formal dan ...
2,Apakah selama sidang KP saya boleh bertanya ke...,"Tidak boleh. Selama sidang, mahasiswa tidak di..."
3,Saya sudah pernah mengambil Seminar semester l...,"Ya, WAJIB mendaftar lagi di D'Office."
4,Berapa minimal entry bimbingan di D'Office unt...,Minimal **15 entry** log/riwayat bimbingan lan...


## 3. Preprocessing

Sebelum model dipakai, semua teks dinormalisasi agar variasi penulisan yang tidak penting tidak mendominasi similarity. Normalisasi ini sengaja dibuat sederhana dan mudah dijelaskan, karena domain proyek adalah FAQ akademik berbahasa Indonesia.

Preprocessing yang dilakukan meliputi:
- lowercase,
- penghapusan tanda baca,
- normalisasi spasi,
- ekspansi singkatan akademik seperti `KP`, `TA`, dan `SKS`.

Alasan utama penggunaan preprocessing ini adalah agar query seperti `sidang kp`, `sidang KP`, dan `sidang kerja praktik` punya peluang lebih besar untuk dipetakan ke makna yang sama.

---


In [26]:
# Kamus singkatan akademik membantu menyamakan variasi bahasa mahasiswa ke bentuk yang lebih konsisten.
SINGKATAN = {
    'kp'  : 'kerja praktik',
    'ta'  : 'tugas akhir',
    'pkl' : 'praktik kerja lapangan',
    'ipk' : 'indeks prestasi kumulatif',
    'nim' : 'nomor induk mahasiswa',
    'nip' : 'nomor induk pegawai',
    'sks' : 'satuan kredit semester',
    'krs' : 'kartu rencana studi',
    'khs' : 'kartu hasil studi',
    'uts' : 'ujian tengah semester',
    'uas' : 'ujian akhir semester',
}

# Normalisasi dibuat sederhana agar mudah dijelaskan di laporan:
# lowercase, hapus tanda baca, rapikan spasi, lalu ekspansi singkatan.
def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r'[^\w\s]', ' ', text)   # hapus tanda baca
    text = re.sub(r'\s+', ' ', text)          # normalkan spasi
    words = text.split()
    words = [SINGKATAN.get(w, w) for w in words]  # ekspansi singkatan
    return ' '.join(words)

# FAQ dibersihkan, dinormalisasi, dan diberi ID agar mudah dilacak saat evaluasi.
faq_df = faq_df.dropna(subset=['question', 'answer']).copy()
faq_df['question_clean'] = faq_df['question'].apply(normalize_text)
faq_df['answer'] = faq_df['answer'].str.strip()
faq_df = faq_df.drop_duplicates(subset=['question_clean']).reset_index(drop=True)
faq_df['faq_id'] = [f'FAQ{idx+1:03d}' for idx in range(len(faq_df))]
faq_df[['faq_id', 'question', 'answer']].head()


,faq_id,question,answer
0,FAQ001,Apa saja yang dinilai oleh supervisor perusaha...,Supervisor perusahaan dan dosen penguji akan m...
1,FAQ002,Bagaimana aturan berpakaian saat sidang KP?,Mahasiswa wajib mengenakan pakaian formal dan ...
2,FAQ003,Apakah selama sidang KP saya boleh bertanya ke...,"Tidak boleh. Selama sidang, mahasiswa tidak di..."
3,FAQ004,Saya sudah pernah mengambil Seminar semester l...,"Ya, WAJIB mendaftar lagi di D'Office."
4,FAQ005,Berapa minimal entry bimbingan di D'Office unt...,Minimal **15 entry** log/riwayat bimbingan lan...


## 4. Dataset Semantic Similarity untuk Fine-Tuning Ringan

Section ini memuat dataset pasangan kalimat yang dipakai untuk menyesuaikan model ke domain akademik proyek. Berbeda dengan `faq.csv` yang berfungsi sebagai knowledge base, file `similarity_pairs.csv` dipakai khusus untuk proses training.

Makna label:
- `1` berarti dua kalimat semakna,
- `0` berarti dua kalimat tidak semakna.

Fine-tuning ringan dipilih karena model dasar multilingual sudah cukup baik, sehingga tujuan training di sini adalah memperbaiki sensitivitas model terhadap phrasing khas mahasiswa, bukan melatih model dari nol.

---


In [27]:
# Muat pasangan semantic similarity yang akan dipakai untuk fine-tuning.
sim_df = pd.read_csv('similarity_pairs.csv')

# Gunakan preprocessing yang sama agar domain training dan retrieval selaras.
sim_df['sentence1'] = sim_df['sentence1'].apply(normalize_text)
sim_df['sentence2'] = sim_df['sentence2'].apply(normalize_text)
sim_df['label'] = sim_df['label'].astype(float)
sim_df


,sentence1,sentence2,label
0,apa saja yang dinilai oleh supervisor perusaha...,apa aja yang dinilai oleh supervisor perusahaa...,1.0
1,apa saja yang dinilai oleh supervisor perusaha...,mohon info saja dinilai oleh supervisor perusa...,1.0
2,apa saja yang dinilai oleh supervisor perusaha...,apakah lulus sidang skripsi berarti langsung l...,0.0
3,apa saja yang dinilai oleh supervisor perusaha...,bagaimana supervisor perusahaan mengisi nilai ...,0.0
4,bagaimana aturan berpakaian saat sidang kerja ...,gimana aturan berpakaian saat sidang kerja pra...,1.0
...,...,...,...
413,apakah mahasiswa bisa mengajukan keberatan ata...,telat upload berkas yudisium denda gak,0.0
414,apakah ada denda atau sanksi jika telat mengun...,telat upload berkas yudisium denda gak,1.0
415,apakah ada denda atau sanksi jika telat mengun...,sanksi kalau telat unggah dokumen yudisium,1.0
416,apakah ada denda atau sanksi jika telat mengun...,cara protes dosen pembimbing skripsi,0.0


## 5. Load Model dan Buat Embedding FAQ

Pada tahap ini, model dasar `paraphrase-multilingual-MiniLM-L12-v2` dimuat sebagai baseline awal. Semua pertanyaan FAQ lalu di-encode menjadi embedding vektor sehingga retrieval dapat dilakukan dengan membandingkan query terhadap embedding FAQ tersebut.

Penting dicatat:
- embedding FAQ baseline dipakai untuk evaluasi awal,
- setelah fine-tuning, embedding FAQ dibangun ulang agar selaras dengan bobot model terbaru.

Section ini adalah fondasi pipeline retrieval, karena semua evaluasi berikutnya bergantung pada kualitas representasi embedding yang dibentuk di sini.

---


In [28]:
# Muat model dasar sebagai baseline awal.
model = SentenceTransformer(BASE_MODEL)

# FAQ di-encode satu kali agar siap dipakai untuk retrieval baseline.
faq_embeddings = model.encode(
    faq_df['question_clean'].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True
)
faq_embeddings.shape


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([111, 384])

## 6. Fungsi Chatbot Retrieval-Based

Section ini berisi fungsi-fungsi inti chatbot. Di sinilah logika semantic retrieval didefinisikan, mulai dari encoding query, perhitungan cosine similarity, sampai pemilihan jawaban terbaik dan evaluasi metrik.

Secara konseptual, alurnya adalah:
1. query pengguna dinormalisasi,
2. query diubah menjadi embedding,
3. cosine similarity terhadap seluruh FAQ dihitung,
4. FAQ dengan skor tertinggi dipilih sebagai kandidat utama,
5. threshold dipakai untuk memutuskan apakah bot cukup yakin untuk menjawab langsung.

Fungsi di section ini dipakai ulang oleh hampir semua section berikutnya, jadi ini adalah inti implementasi notebook.

---


In [29]:
def build_embeddings(active_model):
    """Encode semua FAQ questions menjadi embedding tensor."""
    # Embedding FAQ dibangun ulang setiap kali model berubah, terutama setelah fine-tuning.
    return active_model.encode(
        faq_df['question_clean'].tolist(),
        convert_to_tensor=True,
        normalize_embeddings=True
    )


def retrieve_answer_with_embeddings(user_question, active_model, embeddings,
                                    top_k=3, threshold=THRESHOLD):
    """Cari jawaban terbaik dari FAQ berdasarkan cosine similarity.
    Return: (answer_str, best_score, list_of_top_k_matches)
    """
    # Query dinormalisasi dengan aturan yang sama seperti FAQ.
    query = normalize_text(user_question)
    query_embedding = active_model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_embedding, embeddings)[0]
    top_results = scores.topk(k=min(top_k, len(faq_df)))

    # Simpan kandidat Top-k agar bisa dipakai untuk evaluasi, debug, dan rekomendasi fallback.
    matches = []
    for score, idx in zip(top_results.values.tolist(), top_results.indices.tolist()):
        row = faq_df.iloc[idx]
        matches.append({
            'faq_id'  : row['faq_id'],
            'question': row['question'],
            'answer'  : row['answer'],
            'score'   : float(score)
        })

    best = matches[0]
    # Threshold dipakai sebagai mekanisme keyakinan minimum sebelum bot menjawab langsung.
    if best['score'] < threshold:
        answer = 'Maaf, saya belum menemukan jawaban yang cukup yakin. Silakan lihat top-3 FAQ terdekat.'
    else:
        answer = best['answer']
    return answer, best['score'], matches


# ?? Fungsi evaluasi ??
def top1_accuracy(active_model, active_eval_df, embeddings):
    # Top-1 Accuracy mengecek apakah FAQ target langsung muncul di posisi pertama.
    preds = [retrieve_answer_with_embeddings(r["query"], active_model, embeddings)[2][0]["question"]
             for _, r in active_eval_df.iterrows()]
    trues = active_eval_df["expected_question"].tolist()
    return accuracy_score(trues, preds)


def top3_accuracy(active_model, active_eval_df, embeddings):
    """Jawaban benar ada di antara 3 kandidat teratas."""
    correct = 0
    for _, row in active_eval_df.iterrows():
        _, _, matches = retrieve_answer_with_embeddings(row["query"], active_model, embeddings, top_k=3)
        if any(m["question"] == row["expected_question"] for m in matches):
            correct += 1
    return correct / len(active_eval_df)


def mrr_score(active_model, active_eval_df, embeddings):
    """Mean Reciprocal Rank untuk mengukur rata-rata posisi jawaban benar."""
    total_rr = 0.0
    for _, row in active_eval_df.iterrows():
        _, _, matches = retrieve_answer_with_embeddings(row["query"], active_model, embeddings, top_k=3)
        rr = 0.0
        for rank, m in enumerate(matches, start=1):
            if m["question"] == row["expected_question"]:
                rr = 1.0 / rank
                break
        total_rr += rr
    return total_rr / len(active_eval_df)


def evaluate_retrieval_accuracy(active_model, active_eval_df):
    """Helper singkat untuk evaluasi Top-1 pada model tertentu."""
    emb = build_embeddings(active_model)
    return top1_accuracy(active_model, active_eval_df, emb)


In [30]:
query = 'Bagaimana aturan berpakaian saat sidang KP?'
answer, score, matches = retrieve_answer_with_embeddings(query, model, faq_embeddings)
print('Pertanyaan :', query)
print('Jawaban    :', answer)
print('Skor       :', round(score, 4))
print('Top-3 FAQ  :')
for item in matches:
    print('-', item['question'], '|', round(item['score'], 4))


Pertanyaan : Bagaimana aturan berpakaian saat sidang KP?
Jawaban    : Mahasiswa wajib mengenakan pakaian formal dan **Jaket Kuning** selama sidang.
Skor       : 1.0
Top-3 FAQ  :
- Bagaimana aturan berpakaian saat sidang KP? | 1.0
- Kapan sidang KP dilaksanakan? | 0.6308
- Bagaimana cara mengetahui jadwal sidang KP? | 0.595


## 7. Evaluasi Retrieval dengan `faq_eval_queries.csv`

Section ini mengukur performa retrieval pada query in-domain yang sudah dipetakan ke FAQ target. Evaluasi dilakukan sebelum dan sesudah fine-tuning, sehingga hasilnya bisa dipakai sebagai pembanding kuantitatif yang jelas.

Metrik yang dipakai:
- **Top-1 Accuracy**: seberapa sering jawaban benar langsung ada di posisi pertama,
- **Top-3 Accuracy**: seberapa sering target masih muncul di tiga kandidat teratas,
- **MRR**: seberapa dekat rata-rata posisi jawaban benar dengan posisi teratas.

Section ini penting karena menjadi dasar klaim bahwa fine-tuning memang meningkatkan kualitas retrieval.

---


In [31]:
# Muat query evaluasi in-domain yang sudah memiliki target FAQ benar.
eval_df = pd.read_csv('faq_eval_queries.csv')

# Gunakan embedding FAQ yang sesuai dengan state model saat ini.
# Jika fine-tuning sudah dijalankan, embedding ini otomatis mewakili model terbaru.
emb_eval = build_embeddings(model)

t1  = top1_accuracy(model, eval_df, emb_eval)
t3  = top3_accuracy(model, eval_df, emb_eval)
mrr = mrr_score(model, eval_df, emb_eval)

print('Jumlah query evaluasi :', len(eval_df))
print(f'Top-1 Accuracy        : {t1:.4f}  ? chatbot langsung benar')
print(f'Top-3 Accuracy        : {t3:.4f}  ? jawaban benar ada di 3 kandidat')
print(f'MRR                   : {mrr:.4f}  ? rata-rata posisi jawaban benar')


Jumlah query evaluasi : 222
Top-1 Accuracy        : 0.9099  ← chatbot langsung benar
Top-3 Accuracy        : 0.9640  ← jawaban benar ada di 3 kandidat
MRR                   : 0.9347  ← rata-rata posisi jawaban benar


### Helper Functions

Seluruh helper function dipusatkan di Section 6 agar alur notebook tetap ringkas dan tidak menduplikasi logika yang sama di banyak tempat. Dengan pendekatan ini, perubahan pada logika retrieval cukup dilakukan sekali lalu otomatis berlaku ke evaluasi, error analysis, dan threshold tuning.

---


In [32]:
# Fungsi helper sudah didefinisikan di Section 6.
# Cell ini sengaja dikosongkan untuk menjaga nomor section tetap konsisten.
print('Helper functions: sudah terdefinisi di Section 6.')


Helper functions: sudah terdefinisi di Section 6.


## 8. Fine-Tuning Ringan

Bagian ini melakukan adaptasi domain menggunakan `SentenceTransformerTrainer`. Konfigurasi training sengaja dijaga sederhana agar notebook tetap mudah dipahami dan hasilnya mudah direproduksi.

Alasan konfigurasi utama:
- **2 epoch**: cukup untuk demonstrasi adaptasi domain ringan tanpa training berlebihan,
- **batch size 4**: aman untuk Colab dan dataset relatif kecil,
- **split train/validation**: agar loss dan evaluator dev bisa dipantau,
- **seed 42**: untuk menjaga reproducibility antar-run.

Output utama yang perlu diperhatikan dari section ini adalah train/eval log serta model akhir yang kemudian dievaluasi ulang pada query retrieval.

---


In [33]:
# Acak data lebih dulu agar split train/validation tidak bias urutan file CSV.
sim_df_shuffled = sim_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
dev_size = max(10, int(0.2 * len(sim_df_shuffled)))
train_subset = sim_df_shuffled.iloc[:-dev_size].reset_index(drop=True)
dev_subset   = sim_df_shuffled.iloc[-dev_size:].reset_index(drop=True)

# Trainer menerima data dalam format Dataset dari library datasets.
train_dataset = Dataset.from_dict({
    'sentence1': train_subset['sentence1'].tolist(),
    'sentence2': train_subset['sentence2'].tolist(),
    'label'    : train_subset['label'].astype(float).tolist(),
})
eval_dataset = Dataset.from_dict({
    'sentence1': dev_subset['sentence1'].tolist(),
    'sentence2': dev_subset['sentence2'].tolist(),
    'label'    : dev_subset['label'].astype(float).tolist(),
})

# Konfigurasi training dijaga sederhana agar mudah dipahami dan direproduksi.
batch_size = 4
train_loss = losses.CosineSimilarityLoss(model)
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=dev_subset['sentence1'].tolist(),
    sentences2=dev_subset['sentence2'].tolist(),
    scores=dev_subset['label'].astype(float).tolist(),
    name='dev-similarity'
)
train_steps_per_epoch = max(1, (len(train_dataset) + batch_size - 1) // batch_size)
eval_steps = max(1, train_steps_per_epoch // 2)

args = SentenceTransformerTrainingArguments(
    output_dir='trainer_output',
    num_train_epochs=2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_steps=1,
    eval_strategy='steps',
    eval_steps=eval_steps,
    logging_strategy='steps',
    logging_steps=1,
    save_strategy='no',
    report_to='none',
    seed=SEED,
)

print('Konfigurasi Fine-Tuning')
print('- Model dasar       :', BASE_MODEL)
print('- Loss function     : CosineSimilarityLoss')
print('- Total pasangan    :', len(sim_df))
print('- Train size        :', len(train_subset))
print('- Validation size   :', len(dev_subset))
print('- Epochs            :', args.num_train_epochs)
print('- Batch size        :', batch_size)
print('- Warmup steps      :', args.warmup_steps)
print('- Eval steps        :', args.eval_steps)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

print('Mulai fine-tuning...')
trainer.train()
print('Fine-tuning selesai.')

# Setelah fine-tuning, bangun ulang embedding FAQ agar selaras dengan bobot model terbaru.
faq_embeddings = build_embeddings(model)
post_train_accuracy = evaluate_retrieval_accuracy(model, eval_df)
print(f'Akurasi retrieval setelah fine-tuning: {post_train_accuracy:.4f}')

# Log trainer diringkas ke DataFrame agar mudah dibaca dan dipindahkan ke laporan.
log_history_df = pd.DataFrame(trainer.state.log_history)
train_logs = log_history_df[log_history_df['loss'].notna()][['step', 'epoch', 'loss']].reset_index(drop=True)
eval_logs = log_history_df[log_history_df['eval_loss'].notna()][['step', 'epoch', 'eval_loss']].reset_index(drop=True)
merged_logs = train_logs.merge(eval_logs, on=['step', 'epoch'], how='outer')

print('Jumlah titik training loss :', len(train_logs))
print('Jumlah titik validation loss:', len(eval_logs))
display(merged_logs)


Konfigurasi Fine-Tuning
- Model dasar       : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Loss function     : CosineSimilarityLoss
- Total pasangan    : 418
- Train size        : 335
- Validation size   : 83
- Epochs            : 2
- Batch size        : 4
- Warmup steps      : 1
- Eval steps        : 42


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Mulai fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Dev-similarity Pearson Cosine,Dev-similarity Spearman Cosine
42,0.015063,0.031489,0.934242,0.848301
84,0.106492,0.028015,0.942274,0.837139
126,0.011158,0.029896,0.938249,0.837139
168,0.020782,0.030121,0.938605,0.839169


Fine-tuning selesai.
faq_embeddings diperbarui dengan model fine-tuned.

Log training:


,step,epoch,loss,eval_loss
0,1,0.011905,0.049736,NaN
1,2,0.023810,0.085052,NaN
2,3,0.035714,0.058736,NaN
3,4,0.047619,0.046597,NaN
4,5,0.059524,0.028839,NaN
...,...,...,...,...
168,166,1.976190,0.008077,NaN
169,167,1.988095,0.015101,NaN
170,168,2.000000,0.020782,NaN
171,168,2.000000,NaN,0.030121



Akurasi retrieval setelah fine-tuning: 0.981982


## 9. Ringkasan Hasil Akhir

Section ini merangkum statistik utama notebook dalam satu tabel singkat. Tujuannya adalah memudahkan pembaca memahami skala data dan konfigurasi model sebelum melihat hasil eksperimen yang lebih detail.

Tabel ini cocok dijadikan titik referensi cepat saat menulis laporan atau presentasi.

---


In [34]:
# Ringkasan ini berguna sebagai checkpoint cepat untuk ukuran data dan dimensi model.
summary_df = pd.DataFrame([
    {'komponen': 'Jumlah FAQ', 'nilai': len(faq_df)},
    {'komponen': 'Jumlah Similarity Pairs', 'nilai': len(sim_df)},
    {'komponen': 'Jumlah Query Evaluasi', 'nilai': len(eval_df)},
    {'komponen': 'Dimensi Embedding', 'nilai': int(faq_embeddings.shape[1])},
])

print('Ringkasan Dataset dan Model')
summary_df


Ringkasan Dataset dan Model


,komponen,nilai
0,Jumlah FAQ,111
1,Jumlah Similarity Pairs,418
2,Jumlah Query Evaluasi,222
3,Dimensi Embedding,384


## 10. Evaluasi Sebelum vs Sesudah Fine-Tuning

Di sini baseline model pretrained dibandingkan langsung dengan model setelah fine-tuning. Section ini dibuat agar pembaca bisa melihat dampak fine-tuning secara eksplisit, bukan hanya membaca metrik model akhir tanpa pembanding.

Jika perbedaan akurasi terlihat jelas, maka section ini menjadi bukti utama untuk klaim evaluasi komparatif pada laporan.

---


In [35]:
# Evaluasi komparatif dilakukan dengan memuat ulang model baseline tanpa fine-tuning.
# Dengan cara ini, perbandingan terhadap model saat ini menjadi lebih adil dan eksplisit.
base_model_eval = SentenceTransformer(BASE_MODEL)
base_accuracy = evaluate_retrieval_accuracy(base_model_eval, eval_df)
current_accuracy = evaluate_retrieval_accuracy(model, eval_df)

comparison_df = pd.DataFrame([
    {'model': 'Pretrained SBERT', 'top1_accuracy': base_accuracy},
    {'model': 'Current Model (setelah run notebook)', 'top1_accuracy': current_accuracy},
])

comparison_df


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,top1_accuracy
0,Pretrained SBERT,0.909910
1,Current Model (setelah run notebook),0.981982


## 11. Contoh Hasil Prediksi pada Query Uji

Selain angka agregat, pembaca juga perlu melihat contoh prediksi konkret. Section ini menampilkan sampel query evaluasi, FAQ target, FAQ prediksi, dan skor similarity supaya perilaku model lebih mudah dibayangkan.

Section ini berguna sebagai jembatan antara metrik kuantitatif dan interpretasi kualitatif sebelum masuk ke error analysis.

---


In [36]:
# Ambil sampel query acak agar pembaca bisa melihat perilaku model pada contoh nyata.
current_embeddings = build_embeddings(model)
sample_eval_df = eval_df.sample(5, random_state=SEED).reset_index(drop=True)
rows = []

for _, row in sample_eval_df.iterrows():
    answer, score, matches = retrieve_answer_with_embeddings(row['query'], model, current_embeddings)
    rows.append({
        'query': row['query'],
        'expected_question': row['expected_question'],
        'predicted_question': matches[0]['question'],
        'similarity_score': round(score, 4),
        'match': matches[0]['question'] == row['expected_question']
    })

pd.DataFrame(rows)


,query,expected_question,predicted_question,similarity_score,match
0,apakah benar bisa siswa jurusan ips mengambil ...,Apakah bisa siswa jurusan IPS mengambil prodi ...,Apakah bisa siswa jurusan IPS mengambil prodi ...,0.9961,True
1,surat pengantar magang sebelum ada perusahaan,Apakah mahasiswa bisa mengajukan surat pengant...,Apakah mahasiswa bisa mengajukan surat pengant...,0.9474,True
2,tolong jelaskan lulus sidang skripsi berarti l...,Apakah lulus sidang skripsi berarti langsung l...,Apakah lulus sidang skripsi berarti langsung l...,0.9863,True
3,mohon info saja program studi ada dte,Apa saja program studi yang ada di DTE?,Apa saja program studi yang ada di DTE?,0.9745,True
4,apakah benar di ui tersedia beasiswa,Apakah di UI tersedia beasiswa?,Apakah di UI tersedia beasiswa?,0.9651,True


In [ ]:
# Error analysis aktual untuk query in-domain.
# Cell ini sengaja mengekstrak semua kasus salah agar pembahasan laporan tidak berbasis dugaan.
error_rows = []

for _, row in eval_df.iterrows():
    _, score, matches = retrieve_answer_with_embeddings(
        row['query'], model, current_embeddings, top_k=3
    )

    top3_questions = [m['question'] for m in matches]
    top3_scores = [round(float(m['score']), 4) for m in matches]
    predicted_question = matches[0]['question']

    if predicted_question != row['expected_question']:
        error_rows.append({
            'query': row['query'],
            'expected_question': row['expected_question'],
            'predicted_question': predicted_question,
            'top1_score': round(float(score), 4),
            'expected_in_top3': row['expected_question'] in top3_questions,
            'top3_questions': top3_questions,
            'top3_scores': top3_scores,
        })

error_df = pd.DataFrame(error_rows)
print('Error Analysis Aktual (In-Domain)')
print('Jumlah query salah:', len(error_df))

if error_df.empty:
    print('Tidak ada query yang salah pada evaluasi in-domain ini.')
else:
    display(error_df)

    # Prioritaskan error yang targetnya tidak masuk Top-3, lalu skor salah tertinggi.
    priority_errors = error_df.sort_values(
        by=['expected_in_top3', 'top1_score'], ascending=[True, False]
    ).reset_index(drop=True)

    print('Kasus salah paling representatif (prioritas: target tidak masuk Top-3, lalu skor salah tertinggi):')
    display(priority_errors.head(min(5, len(priority_errors))))


## 12. Evaluasi Komparatif dan Iterasi Pelatihan

Section ini merangkum eksperimen dalam format yang lebih siap pakai untuk laporan. Fokusnya adalah menunjukkan bahwa proyek tidak hanya menjalankan satu model akhir, tetapi juga memiliki pembanding baseline dan iterasi tuning yang terdokumentasi.

Bagian ini sangat relevan untuk pembahasan rubrik bonus evaluasi komparatif.

---


In [37]:
# Ringkas eksperimen baseline vs model utama dalam format tabel yang mudah dipakai di laporan.
comparative_df = pd.DataFrame([
    {
        'run': 'Baseline',
        'model': 'Pretrained SBERT',
        'epochs': 0,
        'batch_size': '-',
        'learning_rate': '-',
        'threshold': THRESHOLD,
        'top1_accuracy': round(base_accuracy, 6),
        'catatan': 'Model awal tanpa fine-tuning'
    },
    {
        'run': 'Main Model',
        'model': 'Sentence-BERT fine-tuned ringan',
        'epochs': 2,
        'batch_size': 4,
        'learning_rate': 'default Sentence-Transformers',
        'threshold': THRESHOLD,
        'top1_accuracy': round(current_accuracy, 6),
        'catatan': 'Model setelah fine-tuning dijalankan di notebook'
    },
])

print('Perbandingan baseline vs model utama')
comparative_df


Perbandingan baseline vs model utama


,run,model,epochs,batch_size,learning_rate,threshold,top1_accuracy,catatan
0,Baseline,Pretrained SBERT,0,-,-,0.55,0.909910,Model awal tanpa fine-tuning
1,Main Model,Sentence-BERT fine-tuned ringan,2,4,default Sentence-Transformers,0.55,0.981982,Model setelah model.fit(...) dijalankan di not...


## 13. Iterasi Terstruktur: Threshold Tuning

Threshold tuning digunakan untuk menjawab kebutuhan praktis yang tidak tercakup oleh Top-1 Accuracy saja, yaitu kapan chatbot sebaiknya menjawab langsung dan kapan sebaiknya menampilkan fallback response.

Penting dipahami bahwa threshold **tidak mengubah ranking FAQ**, tetapi hanya mengubah keputusan akhir berdasarkan tingkat keyakinan model. Karena itu, section ini dipakai untuk melihat trade-off antara keberanian sistem menjawab dan keamanan terhadap query OOD.

---


In [38]:
def evaluate_with_threshold(active_model, active_eval_df, threshold_value):
    # Evaluasi ini hanya mengubah keputusan fallback, bukan ranking semantic retrieval.
    active_embeddings = build_embeddings(active_model)
    preds = []
    trues = []
    fallback_count = 0

    for _, row in active_eval_df.iterrows():
        _, score, matches = retrieve_answer_with_embeddings(
            row['query'], active_model, active_embeddings, threshold=threshold_value
        )
        preds.append(matches[0]['question'])
        trues.append(row['expected_question'])
        fallback_count += int(score < threshold_value)

    return {
        'top1_accuracy': accuracy_score(trues, preds),
        'fallback_rate': fallback_count / len(active_eval_df)
    }

# Muat query out-of-domain dari CSV terpisah agar threshold tuning mudah dijelaskan.
try:
    ood_queries = pd.read_csv('faq_ood_queries.csv')
except FileNotFoundError:
    ood_queries = pd.read_csv('data/raw/faq_ood_queries.csv')

# Gabungkan eval asli dan OOD agar bisa melihat trade-off jawaban vs fallback pada satu tabel.
combined_eval = pd.concat([eval_df, ood_queries], ignore_index=True)

threshold_candidates = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
threshold_rows = []

# Embedding dibangun sekali di luar loop karena threshold tidak mengubah representasi vektor.
active_embeddings = build_embeddings(model)

for th in threshold_candidates:
    preds = []
    trues = []
    fallback_count = 0
    correct_fallback = 0  # Jumlah query OOD yang berhasil ditolak lewat fallback.

    for _, row in combined_eval.iterrows():
        _, score, matches = retrieve_answer_with_embeddings(
            row['query'], model, active_embeddings
        )
        is_ood = (row['expected_question'] == '__OOD__')
        is_fallback = (score < th)

        if not is_ood:
            preds.append(matches[0]['question'])
            trues.append(row['expected_question'])

        fallback_count += int(is_fallback)
        if is_ood and is_fallback:
            correct_fallback += 1

    n_ood = len(ood_queries)
    threshold_rows.append({
        'threshold': th,
        'top1_accuracy': round(accuracy_score(trues, preds), 6),
        'fallback_rate': round(fallback_count / len(combined_eval), 4),
        'ood_detected': f'{correct_fallback}/{n_ood}',
    })

threshold_df = pd.DataFrame(threshold_rows)
print('Threshold Tuning dengan Out-of-Domain Queries')
print(f'Total query evaluasi: {len(eval_df)} in-domain + {len(ood_queries)} out-of-domain')
threshold_df


Threshold Tuning dengan Out-of-Domain Queries
Total query evaluasi: 222 in-domain + 53 out-of-domain


,threshold,top1_accuracy,fallback_rate,ood_detected
0,0.45,0.981982,0.0727,18/53
1,0.50,0.981982,0.1091,28/53
2,0.55,0.981982,0.1455,38/53
3,0.60,0.981982,0.1673,44/53
4,0.65,0.981982,0.1818,48/53
5,0.70,0.981982,0.1855,49/53
6,0.75,0.981982,0.2036,52/53
7,0.80,0.981982,0.2182,52/53


In [ ]:
# Analisis query OOD yang borderline di threshold rendah vs tinggi.
# Tujuannya adalah menunjukkan contoh nyata query yang tampak relevan tetapi seharusnya ditolak.
ood_borderline_rows = []

for _, row in ood_queries.iterrows():
    _, score, matches = retrieve_answer_with_embeddings(
        row['query'], model, active_embeddings, top_k=3, threshold=0.45
    )

    score = round(float(score), 4)
    status_at_0_45 = 'fallback' if score < 0.45 else 'answered'
    status_at_0_75 = 'fallback' if score < 0.75 else 'answered'

    # Simpan query yang berubah status atau tetap lolos pada threshold rendah.
    if status_at_0_45 != status_at_0_75 or status_at_0_45 == 'answered':
        ood_borderline_rows.append({
            'query': row['query'],
            'top1_predicted_question': matches[0]['question'],
            'score': score,
            'status_at_0_45': status_at_0_45,
            'status_at_0_75': status_at_0_75,
        })

ood_borderline_df = pd.DataFrame(ood_borderline_rows)
print('OOD Borderline Analysis')
print('Contoh query OOD yang lolos di threshold rendah atau berubah status di threshold tinggi:', len(ood_borderline_df))

if ood_borderline_df.empty:
    print('Tidak ada query OOD borderline yang perlu dianalisis lebih lanjut.')
else:
    ood_borderline_df = ood_borderline_df.sort_values(
        by=['status_at_0_45', 'status_at_0_75', 'score'],
        ascending=[True, True, False]
    ).reset_index(drop=True)
    display(ood_borderline_df)


## 14. Interpretasi Iterasi

Section ini merangkum temuan threshold tuning dalam bahasa yang lebih interpretatif. Tujuannya adalah membantu pembaca atau audiens presentasi memahami makna dari tabel threshold, bukan hanya membaca angkanya.

Biasanya bagian ini dipakai untuk menjelaskan alasan memilih threshold tertentu, serta kapan threshold rendah atau tinggi lebih cocok secara operasional.

---


In [39]:
# Pilih threshold terbaik berdasarkan akurasi in-domain tertinggi dan fallback rate terendah.
best_threshold_row = threshold_df.sort_values(
    by=['top1_accuracy', 'fallback_rate'], ascending=[False, True]
).iloc[0]

print('Kesimpulan evaluasi komparatif dan iterasi')
print(f'- Baseline accuracy      : {base_accuracy:.6f}')
print(f'- Current model accuracy : {current_accuracy:.6f}')
print(f'- Kenaikan accuracy      : {current_accuracy - base_accuracy:.6f}')
print(f"- Threshold terbaik      : {best_threshold_row['threshold']:.2f}")
print(f"- Fallback rate          : {best_threshold_row['fallback_rate']:.4f}")
print(f"- OOD terdeteksi         : {best_threshold_row['ood_detected']}")
print()
print('Analisis:')
print('- Threshold rendah (0.45-0.55): bot lebih sering menjawab, termasuk pada query di luar topik')
print('- Threshold tinggi (0.70-0.80): bot lebih selektif dan lebih banyak query OOD yang ditolak')
print('- Trade-off: threshold lebih tinggi = lebih aman tapi bisa menolak pertanyaan valid')


Kesimpulan evaluasi komparatif dan iterasi
- Baseline accuracy      : 0.909910
- Current model accuracy : 0.981982
- Kenaikan accuracy      : 0.072072
- Threshold terbaik      : 0.45
- Fallback rate          : 0.0727
- OOD terdeteksi         : 18/53

Analisis:
- Threshold rendah (0.45-0.55): bot selalu menjawab, termasuk pertanyaan di luar topik
- Threshold tinggi (0.70-0.80): bot lebih selektif, OOD query di-fallback dengan benar
- Trade-off: threshold lebih tinggi = lebih aman tapi bisa menolak pertanyaan valid


## 15. Simpan Model untuk Deployment (Streamlit)

Section terakhir menyiapkan artefak deployment sederhana. Model fine-tuned, embedding FAQ, dan metadata penting disimpan agar chatbot bisa dipanggil lagi tanpa mengulang proses training dari awal.

Alasan section ini penting:
- memudahkan demo lanjutan di Streamlit,
- memisahkan tahap training dari tahap inference,
- menunjukkan bahwa notebook tidak berhenti di eksperimen, tetapi juga menyiapkan artefak yang bisa dipakai ulang.

---


In [40]:
import pickle
import os

# 1. Simpan model fine-tuned.
# Folder model ini bisa dipakai kembali untuk inference tanpa retraining.
MODEL_SAVE_PATH = 'faq_sbert_finetuned'
model.save(MODEL_SAVE_PATH)
print(f'Model disimpan ke: {MODEL_SAVE_PATH}/')
print(f'  Ukuran folder: {sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(MODEL_SAVE_PATH) for f in fn) / 1e6:.1f} MB')

# 2. Pre-compute embedding FAQ dan simpan bersama data.
# Embedding FAQ disimpan agar aplikasi deployment bisa langsung melakukan retrieval.
faq_embeddings_np = model.encode(
    faq_df['question_clean'].tolist(),
    normalize_embeddings=True
)

export_data = {
    'questions': faq_df['question'].tolist(),
    'answers': faq_df['answer'].tolist(),
    'questions_clean': faq_df['question_clean'].tolist(),
    'faq_ids': faq_df['faq_id'].tolist(),
    'embeddings': faq_embeddings_np,
    'threshold': THRESHOLD,
}

with open('faq_data.pkl', 'wb') as f:
    pickle.dump(export_data, f)

print(f'Data FAQ disimpan ke: faq_data.pkl')
print(f'  Jumlah FAQ   : {len(export_data["questions"])}')
print(f'  Dimensi emb  : {faq_embeddings_np.shape}')
print(f'  Threshold    : {THRESHOLD}')

# 3. Zip model untuk download.
# Arsip zip memudahkan pemindahan artefak dari Colab ke local machine atau Streamlit app.
!zip -r faq_sbert_finetuned.zip faq_sbert_finetuned/
print()
print('Semua file siap! Untuk download:')
print('   from google.colab import files')
print('   files.download("faq_sbert_finetuned.zip")')
print('   files.download("faq_data.pkl")')
print('   files.download("faq.csv")')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model disimpan ke: faq_sbert_finetuned/
  Ukuran folder: 487.8 MB
Data FAQ disimpan ke: faq_data.pkl
  Jumlah FAQ   : 111
  Dimensi emb  : (111, 384)
  Threshold    : 0.55
updating: faq_sbert_finetuned/ (stored 0%)
updating: faq_sbert_finetuned/sentence_bert_config.json (deflated 43%)
updating: faq_sbert_finetuned/config_sentence_transformers.json (deflated 40%)
updating: faq_sbert_finetuned/README.md (deflated 78%)
updating: faq_sbert_finetuned/model.safetensors (deflated 8%)
updating: faq_sbert_finetuned/config.json (deflated 53%)
updating: faq_sbert_finetuned/tokenizer_config.json (deflated 55%)
updating: faq_sbert_finetuned/tokenizer.json (deflated 76%)
updating: faq_sbert_finetuned/modules.json (deflated 55%)
updating: faq_sbert_finetuned/1_Pooling/ (stored 0%)
updating: faq_sbert_finetuned/1_Pooling/config.json (deflated 16%)

Semua file siap! Untuk download:
   from google.colab import files
   files.download("faq_sbert_finetuned.zip")
   files.download("faq_data.pkl")
   files.

## Kesimpulan

Section penutup ini merangkum poin-poin teknis utama notebook: model yang dipakai, metode retrieval, proses fine-tuning, evaluasi, serta tuning threshold. Dengan demikian, pembaca yang ingin melihat ringkasan cepat tetap bisa memahami keseluruhan pipeline tanpa membaca ulang seluruh notebook dari atas.

---
